# 04 — Defense

Two defense strategies against adversarial attacks on the IDS model:

| Defense | How it works |
|---------|-------------|
| **Adversarial Training** | Retrain on mix of clean + adversarial data |
| **Feature Squeezing** | Quantise features to remove fine perturbations |

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Setup

In [ ]:
from src.data_pipeline import load_data, auto_detect_dataset, get_feature_bounds
from src.model import train_random_forest, evaluate
from src.attacker import train_surrogate, fgsm_attack, evaluate_attack
from sklearn.ensemble import RandomForestClassifier

data_path, _ = auto_detect_dataset()
X_train, X_test, y_train, y_test, scaler = load_data(data_path, n_features=20)
X_min, X_max = get_feature_bounds(X_train)

rf = train_random_forest(X_train, y_train)
surrogate = train_surrogate(X_train, y_train)

baseline_dr = evaluate(rf, X_test, y_test)['detection_rate']
print(f'Baseline detection rate: {baseline_dr*100:.1f}%')

# Attack to defend against
X_adv_test = fgsm_attack(surrogate, X_test, y_test, epsilon=0.1, X_min=X_min, X_max=X_max)
attack_dr = evaluate_attack(rf, X_adv_test, y_test)
print(f'Under attack (ε=0.1): {attack_dr*100:.1f}%')

## Defense 1 — Adversarial Training

Augment training data with FGSM adversarial examples and retrain.
The model learns to handle perturbed inputs during training.

In [ ]:
from src.defender import adversarial_training

print('Training robust model (this takes a moment)...')
robust_rf = adversarial_training(
    RandomForestClassifier, X_train, y_train, surrogate,
    epsilon=0.1, X_min=X_min, X_max=X_max,
    n_estimators=100, max_depth=10, n_jobs=-1,
)

adv_train_dr_clean = evaluate(robust_rf, X_test, y_test)['detection_rate']
adv_train_dr_attack = evaluate_attack(robust_rf, X_adv_test, y_test)

print(f'Robust model — clean accuracy  : {adv_train_dr_clean*100:.1f}%')
print(f'Robust model — under attack    : {adv_train_dr_attack*100:.1f}%')
print(f'Recovery vs undefended attack  : +{(adv_train_dr_attack - attack_dr)*100:.1f}pp')

## Defense 2 — Feature Squeezing

Quantise each feature to 8-bit precision before inference.
Fine-grained adversarial perturbations are rounded away.
**No retraining required** — purely post-processing.

In [ ]:
from src.defender import feature_squeezing_detection_rate, round_to_bits

for bits in [4, 6, 8, 10, 16]:
    dr = feature_squeezing_detection_rate(rf, X_adv_test, y_test, n_bits=bits)
    print(f'  {bits:2d}-bit squeezing → {dr*100:.1f}% detection')

## Comparison

In [ ]:
from src.defender import feature_squeezing_detection_rate
squeeze_dr = feature_squeezing_detection_rate(rf, X_adv_test, y_test, n_bits=8)

labels = ['Baseline\n(no attack)', 'Under FGSM\n(ε=0.1)',
          'Adv. Training', 'Feat. Squeezing']
values = [baseline_dr*100, attack_dr*100, adv_train_dr_attack*100, squeeze_dr*100]
colors = ['#27ae60','#e74c3c','#2980b9','#8e44ad']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, values, color=colors, edgecolor='black', width=0.55)
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
            f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel('Detection Rate (%)')
ax.set_title('Defense Effectiveness — AdverScan')
ax.set_ylim(0, 115)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'plots', 'defense_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

## Discussion

**Key observations:**
- Both defenses partially recover detection rate
- Adversarial training is more effective but requires retraining
- Feature squeezing is zero-cost but less powerful
- Neither fully restores baseline performance — there is always a residual vulnerability

**Trade-offs:**
- Adversarial training can reduce clean accuracy slightly (model sees noisier data)
- Feature squeezing may lose detail in legitimate traffic features
- Ensemble defense (not shown here) combines both strengths